# Data Visualization

**Authors:** Benedikt Prisett & Stijn Diemel

This notebook presents the visualizations that have been created to answer the provided questions. For each question we also include some of the earlier design iterations and then present the final visualization, that we ultimately used in the final multi-view dashboard.
For the data cleaning process please view the notebook: [02_data_cleaning.ipynb](./02_cleaning.ipynb).


## Setup


In [1]:
import pandas as pd
import altair as alt

In [2]:
df = pd.read_csv("../data/clean_data/simpsons_script_lines_clean.csv")
alt.data_transformers.enable("vegafusion")
df.head(10).style.set_properties(subset=["spoken_words"], **{"white-space": "pre-wrap"})

,episode_id,season,number_in_season,title,imdb_rating,line_number,timestamp_in_ms,character,location_id,location,spoken_words,word_count,sentence_count
0,1,1,1,Simpsons Roasting on an Open Fire,8.200000,2,8000,Marge Simpson,2,Car,"Ooo, careful, Homer.",3,1
1,1,1,1,Simpsons Roasting on an Open Fire,8.200000,3,10000,Homer Simpson,2,Car,There's no time to be careful.,6,1
2,1,1,1,Simpsons Roasting on an Open Fire,8.200000,4,10000,Homer Simpson,2,Car,We're late.,2,1
3,1,1,1,Simpsons Roasting on an Open Fire,8.200000,7,24000,Marge Simpson,4,Auditorium,"Sorry, Excuse us. Pardon me...",5,2
4,1,1,1,Simpsons Roasting on an Open Fire,8.200000,8,26000,Homer Simpson,4,Auditorium,"Hey, Norman. How's it going? So you got dragged down here, too... heh, heh. How ya doing, Fred? Excuse me, Fred.",21,6
5,1,1,1,Simpsons Roasting on an Open Fire,8.200000,9,34000,Homer Simpson,4,Auditorium,Pardon my galoshes.,3,1
6,1,1,1,Simpsons Roasting on an Open Fire,8.200000,10,44000,Seymour Skinner,4,Auditorium,"Wasn't that wonderful? And now, ""Santas of Many Lands,"" as presented by the entire second grade class.",17,2
7,1,1,1,Simpsons Roasting on an Open Fire,8.200000,11,55000,Marge Simpson,4,Auditorium,Oh... Lisa's class.,3,2
8,1,1,1,Simpsons Roasting on an Open Fire,8.200000,12,57000,JANEY,4,Auditorium,"Frohlich weihnachten -- that's German for Merry Christmas. In Germany, Santa's servant Ruprecht gives presents to good children and whipping rods to the parents of bad ones.",27,2
9,1,1,1,Simpsons Roasting on an Open Fire,8.200000,13,75000,Todd Flanders,4,Auditorium,"Meri Kurimasu. I am Hotseiosha, a Japanese priest who acts like Santa Claus. I have eyes in the back of my head so children better behave when I'm nearby.",29,3


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 131710 entries, 0 to 131709
Data columns (total 13 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   episode_id        131710 non-null  int64  
 1   season            131710 non-null  int64  
 2   number_in_season  131710 non-null  int64  
 3   title             131710 non-null  object 
 4   imdb_rating       131710 non-null  float64
 5   line_number       131710 non-null  int64  
 6   timestamp_in_ms   131710 non-null  int64  
 7   character         131710 non-null  object 
 8   location_id       131710 non-null  int64  
 9   location          131710 non-null  object 
 10  spoken_words      131710 non-null  object 
 11  word_count        131710 non-null  int64  
 12  sentence_count    131710 non-null  int64  
dtypes: float64(1), int64(8), object(4)
memory usage: 13.1+ MB


---

## Question 1: What are the characters that issue more words, and how are the word numbers distributed?


### Design Iterations


In [4]:
# Q1 - iteration 2
words_per_char = df.groupby("character", as_index=False)["word_count"].sum()
q1_words_table = (
    words_per_char.rename(columns={"word_count": "total_word_count"})
    .sort_values("total_word_count", ascending=False)
    .assign(percentage=lambda d: d["total_word_count"] / d["total_word_count"].sum())
    .reset_index(drop=True)
)

print(f"Overall length of q1_words_table: {len(q1_words_table)}")

Overall length of q1_words_table: 6248


In [5]:
q1_words_table.head(15).style.format(
    {"total_word_count": "{:,.0f}", "percentage": "{:.2%}"}
)

,character,total_word_count,percentage
0,Homer Simpson,"270,691",20.79%
1,Marge Simpson,"124,456",9.56%
2,Bart Simpson,"109,011",8.37%
3,Lisa Simpson,"99,110",7.61%
4,C. Montgomery Burns,"36,212",2.78%
5,Moe Szyslak,"32,835",2.52%
6,Seymour Skinner,"28,069",2.16%
7,Ned Flanders,"22,880",1.76%
8,Krusty the Clown,"20,435",1.57%
9,Chief Wiggum,"19,992",1.54%


In [6]:
q1_words_table.tail().style.format(
    {"total_word_count": "{:,.0f}", "percentage": "{:.2%}"}
)

,character,total_word_count,percentage
6243,Everyone Else,1,0.00%
6244,BUCK/TABITHA,1,0.00%
6245,BURLY BARTENDER,1,0.00%
6246,KRUSTY'S HEAD,1,0.00%
6247,Ostrich,0,0.00%


In [7]:
# Q1 - iteration 2.1
def get_top_n(n=20):
    return words_per_char.nlargest(n, "word_count")


n = 10
top_n = get_top_n(n)

bars = (
    alt.Chart(top_n)
    .mark_bar()
    .encode(
        x=alt.X("word_count:Q", title="Total Word Count"),
        y=alt.Y("character:N", sort="-x", title="Character"),
        tooltip=["character", "word_count"],
    )
)

labels = (
    alt.Chart(top_n)
    .mark_text(align="left", baseline="middle", dx=4)
    .encode(
        x="word_count:Q",
        y=alt.Y("character:N", sort="-x"),
        text=alt.Text("word_count:Q", format=","),
    )
)

(bars + labels).properties(
    title=f"Top {n} Characters by Total Word Count", width=500, height=400
)

alt.LayerChart(...)

In [8]:
# Q1 - iteration 2.2
n = 10
top_n = get_top_n(n)

bars = (
    alt.Chart(top_n)
    .mark_bar()
    .encode(
        x=alt.X(
            "character:N",
            sort="-y",
            title="Character",
            axis=alt.Axis(labelAngle=-45),
        ),
        y=alt.Y("word_count:Q", title="Total Word Count"),
        tooltip=["character", alt.Tooltip("word_count:Q", format=",")],
    )
)

bars.properties(title=f"Top {n} Characters by Total Word Count", width=600, height=400)


alt.Chart(...)

In [9]:
# Q1 - iteration 3
n = 40
base = alt.Chart(get_top_n(n)).encode(
    x=alt.X("word_count:Q", title="Total Word Count"),
    y=alt.Y("character:N", sort="-x", title="Character"),
    tooltip=["character", "word_count"],
)

points = base.mark_circle(size=80, color="#e45756")
lines = base.mark_rule(color="#e45756")

(lines + points).properties(
    title=f"Top {n} Characters by Total Word Count", width=500, height=400
)

alt.LayerChart(...)

In [34]:
# Q1 - iteration 4: horizontal box plot of per-episode word counts per character
n = 10
top_characters = df.groupby("character")["word_count"].sum().nlargest(n).index.tolist()

per_episode_words = (
    df[df["character"].isin(top_characters)]
    .groupby(["episode_id", "character"], as_index=False)["word_count"]
    .sum()
)

alt.Chart(per_episode_words).mark_boxplot().encode(
    x=alt.X("word_count:Q", title="Words per Episode"),
    y=alt.Y("character:N", sort=top_characters, title="Character"),
    color=alt.Color("character:N", legend=None),
    tooltip=[
        "character",
        alt.Tooltip("word_count:Q", format=",", title="Words"),
    ],
).properties(
    title=f"Per-Episode Word Count Distribution — Top {n} Characters",
    width=600,
    height=400,
)


alt.Chart(...)

- n should be reasonable default
- user can choose to show larger n if they wish (this would possible also effect other charts in q2 and q5)
- should be possible that one (or two) characters are clickable and then show more details and also trigger actions in other charts (https://altair-viz.github.io/gallery/interactive_bar_select_highlight.html)
- do these charts show the distribution or how do we address this?
- should use an overall consistent color scheme


### Final Visualization


In [10]:
# Q1 - final

### Design Discussion (max 200 words)

_TODO: Explain chart type choice, design process steps, changes for legibility, how interaction answers the question, and alternatives considered._


---

## Question 2: How have the word numbers among the characters evolved throughout the available seasons?


### Design Iterations


In [11]:
# Q2 - iteration 1: line chart of word count per season for top n characters
n = 5
top_characters = df.groupby("character")["word_count"].sum().nlargest(n).index.tolist()

words_per_season = (
    df[df["character"].isin(top_characters)]
    .groupby(["season", "character"], as_index=False)["word_count"]
    .sum()
)

alt.Chart(words_per_season).mark_line(point=True).encode(
    x=alt.X("season:O", title="Season", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("word_count:Q", title="Total Word Count"),
    color=alt.Color("character:N", title="Character"),
    tooltip=["character", "season", alt.Tooltip("word_count:Q", format=",")],
).properties(
    title=f"Word Count per Season — Top {n} Characters",
    width=700,
    height=400,
)

alt.Chart(...)

In [12]:
# Q2 - iteration 1.2: proportion of words per season for top n characters
n = 5
top_characters = df.groupby("character")["word_count"].sum().nlargest(n).index.tolist()

words_per_season_all = (
    df.groupby("season", as_index=False)["word_count"]
    .sum()
    .rename(columns={"word_count": "season_total"})
)

words_per_season_top = (
    df[df["character"].isin(top_characters)]
    .groupby(["season", "character"], as_index=False)["word_count"]
    .sum()
    .merge(words_per_season_all, on="season")
    .assign(proportion=lambda d: d["word_count"] / d["season_total"])
)

alt.Chart(words_per_season_top).mark_line(point=True).encode(
    x=alt.X("season:O", title="Season", axis=alt.Axis(labelAngle=0)),
    y=alt.Y(
        "proportion:Q", title="Proportion of Total Words", axis=alt.Axis(format="%")
    ),
    color=alt.Color("character:N", title="Character"),
    tooltip=["character", "season", alt.Tooltip("proportion:Q", format=".1%")],
).properties(
    title=f"Proportion of Words per Season — Top {n} Characters",
    width=700,
    height=400,
)

alt.Chart(...)

In [13]:
# Q2 - iteration 2: small multiples line chart per character
n = 8
top_characters = df.groupby("character")["word_count"].sum().nlargest(n).index.tolist()

words_per_season = (
    df[df["character"].isin(top_characters)]
    .groupby(["season", "character"], as_index=False)["word_count"]
    .sum()
)

alt.Chart(words_per_season).mark_line(point=True).encode(
    x=alt.X("season:Q", title="Season", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("word_count:Q", title="Word Count"),
    color=alt.Color("character:N", legend=None),
).properties(
    width=200,
    height=120,
).facet(
    facet=alt.Facet(
        "character:N",
        title=None,
        sort=top_characters,
        header=alt.Header(labelPadding=2, labelFontSize=12),
    ),
    columns=4,
).properties(
    title=alt.TitleParams(
        text=f"Word Count per Season — Top {n} Characters",
        anchor="start",
        # offset=5,
    ),
)

alt.FacetChart(...)

- decision between total and proportion
- if small multiples there could be interaction between selecting top n in q1 and what is shown here (a bit tricky with the spacing; could have a max here e.g. only show top 8 or 6 or 9 etc)
- also selecting one or two should then highlight them here


### Final Visualization


In [14]:
# Q2 - final

### Design Discussion (max 200 words)

_TODO: Explain chart type choice, design process steps, changes for legibility, how interaction answers the question, and alternatives considered._


---

## Question 3: Compare the word distribution for a pair of characters, for a selected season


- maybe here we could use the timeline example from the VOTD week 12:


### Design Iterations

Word frequency and TF-IDF based approaches were explored in [05_word_frequency.ipynb](./05_word_frequency.ipynb). Below we take a different approach focusing on the overall development of word count across the season.


In [15]:
# Q3 - iteration 1: dual line chart — word count per episode for two characters in a season
character_a = "Homer Simpson"
character_b = "Marge Simpson"
season = 5

season_df = (
    df[df["season"] == season]
    .groupby(["number_in_season", "character"], as_index=False)["word_count"]
    .sum()
)

pair_df = season_df[season_df["character"].isin([character_a, character_b])]

alt.Chart(pair_df).mark_line(point=True).encode(
    x=alt.X("number_in_season:O", title="Episode", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("word_count:Q", title="Word Count"),
    color=alt.Color("character:N", title="Character"),
    tooltip=["character", "number_in_season", alt.Tooltip("word_count:Q", format=",")],
).properties(
    title=f"Word Count per Episode — {character_a} vs {character_b}, Season {season}",
    width=650,
    height=350,
)

alt.Chart(...)

In [16]:
# Q3 - iteration 2: mirrored/diverging bar chart — one character up, one down
character_a = "Homer Simpson"
character_b = "Marge Simpson"
season = 5

season_df = (
    df[df["season"] == season]
    .groupby(["number_in_season", "character"], as_index=False)["word_count"]
    .sum()
)

pair_df = season_df[season_df["character"].isin([character_a, character_b])].copy()
pair_df["word_count_signed"] = pair_df.apply(
    lambda r: r["word_count"] if r["character"] == character_a else -r["word_count"],
    axis=1,
)

alt.Chart(pair_df).mark_bar().encode(
    x=alt.X("number_in_season:O", title="Episode", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("word_count_signed:Q", title="Word Count"),
    color=alt.Color(
        "character:N",
        title="Character",
        scale=alt.Scale(
            domain=[character_a, character_b], range=["#4c78a8", "#e45756"]
        ),
    ),
    tooltip=["character", "number_in_season", alt.Tooltip("word_count:Q", format=",")],
).properties(
    title=f"Word Count per Episode (Diverging) — {character_a} vs {character_b}, Season {season}",
    width=650,
    height=350,
)

alt.Chart(...)

In [17]:
# Q3 - iteration 3: stacked area chart with proportion of words per episode
character_a = "Homer Simpson"
character_b = "Marge Simpson"
season = 5

season_df = (
    df[df["season"] == season]
    .groupby(["number_in_season", "character"], as_index=False)["word_count"]
    .sum()
)

pair_df = season_df[season_df["character"].isin([character_a, character_b])].copy()

# Compute proportion within each episode (only among the two selected characters)
episode_totals = pair_df.groupby("number_in_season")["word_count"].transform("sum")
pair_df["proportion"] = pair_df["word_count"] / episode_totals

alt.Chart(pair_df).mark_area().encode(
    x=alt.X("number_in_season:O", title="Episode", axis=alt.Axis(labelAngle=0)),
    y=alt.Y(
        "proportion:Q",
        title="Proportion of Words",
        stack="normalize",
        axis=alt.Axis(format="%"),
    ),
    color=alt.Color(
        "character:N",
        title="Character",
        scale=alt.Scale(
            domain=[character_a, character_b], range=["#4c78a8", "#e45756"]
        ),
    ),
    tooltip=[
        "character",
        "number_in_season",
        alt.Tooltip("proportion:Q", format=".0%"),
        alt.Tooltip("word_count:Q", format=",", title="Words"),
    ],
).properties(
    title=f"Proportion of Words per Episode — {character_a} vs {character_b}, Season {season}",
    width=650,
    height=350,
)

alt.Chart(...)

In [18]:
# Q3 - iteration 4: normalized stacked bar chart — proportion per episode
character_a = "Homer Simpson"
character_b = "Marge Simpson"
season = 5

season_df = (
    df[df["season"] == season]
    .groupby(["number_in_season", "character"], as_index=False)["word_count"]
    .sum()
)

pair_df = season_df[season_df["character"].isin([character_a, character_b])].copy()

episode_totals = pair_df.groupby("number_in_season")["word_count"].transform("sum")
pair_df["proportion"] = pair_df["word_count"] / episode_totals

alt.Chart(pair_df).mark_bar().encode(
    x=alt.X("number_in_season:O", title="Episode", axis=alt.Axis(labelAngle=0)),
    y=alt.Y(
        "proportion:Q",
        title="Proportion of Words",
        stack="normalize",
        axis=alt.Axis(format="%"),
    ),
    color=alt.Color(
        "character:N",
        title="Character",
        scale=alt.Scale(
            domain=[character_a, character_b], range=["#4c78a8", "#e45756"]
        ),
    ),
    tooltip=[
        "character",
        "number_in_season",
        alt.Tooltip("proportion:Q", format=".0%"),
        alt.Tooltip("word_count:Q", format=",", title="Words"),
    ],
).properties(
    title=f"Proportion of Words per Episode — {character_a} vs {character_b}, Season {season}",
    width=650,
    height=350,
)

alt.Chart(...)

In [19]:
# Q3 - iteration 5: grouped bar chart — word count per episode, two characters side by side
character_a = "Homer Simpson"
character_b = "Marge Simpson"
season = 5

season_df = (
    df[df["season"] == season]
    .groupby(["number_in_season", "character"], as_index=False)["word_count"]
    .sum()
)

pair_df = season_df[season_df["character"].isin([character_a, character_b])].copy()

# Ensure every (episode, character) combination exists so bars align even if a
# character has no lines in an episode.
all_episodes = pd.DataFrame(
    {"number_in_season": sorted(pair_df["number_in_season"].unique())}
)
all_characters = pd.DataFrame({"character": [character_a, character_b]})
pair_df = (
    all_episodes.merge(all_characters, how="cross")
    .merge(pair_df, on=["number_in_season", "character"], how="left")
    .fillna({"word_count": 0})
)
pair_df["word_count"] = pair_df["word_count"].astype(int)

alt.Chart(pair_df).mark_bar().encode(
    x=alt.X("number_in_season:O", title="Episode", axis=alt.Axis(labelAngle=0)),
    xOffset=alt.XOffset("character:N", sort=[character_a, character_b]),
    y=alt.Y("word_count:Q", title="Word Count"),
    color=alt.Color(
        "character:N",
        title="Character",
        scale=alt.Scale(
            domain=[character_a, character_b], range=["#4c78a8", "#e45756"]
        ),
    ),
    tooltip=[
        "character",
        alt.Tooltip("number_in_season:O", title="Episode"),
        alt.Tooltip("word_count:Q", format=","),
    ],
).properties(
    title=f"Word Count per Episode — {character_a} vs {character_b}, Season {season}",
    width=650,
    height=350,
)


alt.Chart(...)

### Final Visualization


In [20]:
# Q3 - final

### Design Discussion (max 200 words)

_TODO: Explain chart type choice, design process steps, changes for legibility, how interaction answers the question, and alternatives considered._


---

## Question 4: Compare the word distribution for a pair of characters, over a concrete episode


### Design Iterations


In [21]:
# Q4 - iteration 1: cumulative word count over line number within an episode
character_a = "Homer Simpson"
character_b = "Marge Simpson"
season = 5
episode = 1

episode_df = df[
    (df["season"] == season)
    & (df["number_in_season"] == episode)
    & (df["character"].isin([character_a, character_b]))
].copy()

# Compute cumulative word count per character ordered by line number
episode_df = episode_df.sort_values("line_number")
episode_df["cumulative_words"] = episode_df.groupby("character")["word_count"].cumsum()

alt.Chart(episode_df).mark_line().encode(
    x=alt.X("line_number:Q", title="Line Number"),
    y=alt.Y("cumulative_words:Q", title="Cumulative Word Count"),
    color=alt.Color(
        "character:N",
        title="Character",
        scale=alt.Scale(
            domain=[character_a, character_b], range=["#4c78a8", "#e45756"]
        ),
    ),
    tooltip=[
        "character",
        "line_number",
        alt.Tooltip("cumulative_words:Q", format=","),
        "spoken_words",
    ],
).properties(
    title=f"Cumulative Words — {character_a} vs {character_b}, S{season}E{episode}",
    width=650,
    height=350,
)

alt.Chart(...)

In [22]:
# Q4 - iteration 3: binned normalized stacked bar — proportion per episode segment
character_a = "Homer Simpson"
character_b = "Marge Simpson"
season = 5
episode = 1
n_bins = 25

episode_df = df[
    (df["season"] == season)
    & (df["number_in_season"] == episode)
    & (df["character"].isin([character_a, character_b]))
].copy()

episode_df = episode_df.sort_values("line_number")

# Bin line numbers into equal segments (use integer labels for correct ordering)
episode_df["segment"] = pd.cut(
    episode_df["line_number"],
    bins=n_bins,
    labels=list(range(1, n_bins + 1)),
)
episode_df["segment"] = episode_df["segment"].astype(int)

segment_df = episode_df.groupby(["segment", "character"], as_index=False)[
    "word_count"
].sum()

# Ensure all segment-character combinations exist (fill missing with 0)
all_segments = pd.DataFrame({"segment": range(1, n_bins + 1)})
all_characters = pd.DataFrame({"character": [character_a, character_b]})
full_index = all_segments.merge(all_characters, how="cross")
segment_df = full_index.merge(segment_df, on=["segment", "character"], how="left")
segment_df["word_count"] = segment_df["word_count"].fillna(0).astype(int)

segment_totals = segment_df.groupby("segment")["word_count"].transform("sum")
segment_df["proportion"] = segment_df["word_count"] / segment_totals.replace(0, 1)

alt.Chart(segment_df).mark_bar().encode(
    x=alt.X("segment:O", title="Episode Segment", axis=alt.Axis(labelAngle=0)),
    y=alt.Y(
        "proportion:Q",
        title="Proportion of Words",
        stack="normalize",
        axis=alt.Axis(format="%"),
    ),
    color=alt.Color(
        "character:N",
        title="Character",
        scale=alt.Scale(
            domain=[character_a, character_b], range=["#4c78a8", "#e45756"]
        ),
    ),
    tooltip=[
        "character",
        "segment",
        alt.Tooltip("proportion:Q", format=".0%"),
        alt.Tooltip("word_count:Q", format=",", title="Words"),
    ],
).properties(
    title=f"Word Proportion per Segment — {character_a} vs {character_b}, S{season}E{episode}",
    width=650,
    height=350,
)

alt.Chart(...)

In [ ]:
# Q4 - iteration 4: grouped bar chart — word count per minute, two characters side by side
character_a = "Homer Simpson"
character_b = "Marge Simpson"
season = 5
episode = 1

episode_df = df[
    (df["season"] == season)
    & (df["number_in_season"] == episode)
    & (df["character"].isin([character_a, character_b]))
].copy()

# Convert ms timestamp to integer minute index (0 = first minute).
episode_df["minute"] = (episode_df["timestamp_in_ms"] // 60_000).astype(int)

minute_df = episode_df.groupby(["minute", "character"], as_index=False)[
    "word_count"
].sum()

# Ensure every (minute, character) combination exists so bars align even if a
# character has no lines in a given minute.
all_minutes = pd.DataFrame(
    {
        "minute": range(
            int(episode_df["minute"].min()), int(episode_df["minute"].max()) + 1
        )
    }
)
all_characters = pd.DataFrame({"character": [character_a, character_b]})
minute_df = (
    all_minutes.merge(all_characters, how="cross")
    .merge(minute_df, on=["minute", "character"], how="left")
    .fillna({"word_count": 0})
)
minute_df["word_count"] = minute_df["word_count"].astype(int)

alt.Chart(minute_df).mark_bar().encode(
    x=alt.X("minute:O", title="Minute", axis=alt.Axis(labelAngle=0)),
    xOffset=alt.XOffset("character:N", sort=[character_a, character_b]),
    y=alt.Y("word_count:Q", title="Word Count"),
    color=alt.Color(
        "character:N",
        title="Character",
        scale=alt.Scale(
            domain=[character_a, character_b], range=["#4c78a8", "#e45756"]
        ),
    ),
    tooltip=[
        "character",
        alt.Tooltip("minute:O", title="Minute"),
        alt.Tooltip("word_count:Q", format=","),
    ],
).properties(
    title=f"Word Count per Minute — {character_a} vs {character_b}, S{season}E{episode}",
    width=650,
    height=350,
)


alt.Chart(...)

### Final Visualization


In [24]:
# Q4 - final

### Design Discussion (max 200 words)

_TODO: Explain chart type choice, design process steps, changes for legibility, how interaction answers the question, and alternatives considered._


---

## Question 5: What are the characters that issue more sentences, and how are the sentence numbers distributed?


### Design Iterations


In [25]:
# Q5 - iteration 1
sentences_per_char = df.groupby("character", as_index=False)["sentence_count"].sum()
q5_sentences_table = (
    sentences_per_char.rename(columns={"sentence_count": "total_sentence_count"})
    .sort_values("total_sentence_count", ascending=False)
    .assign(
        percentage=lambda d: d["total_sentence_count"] / d["total_sentence_count"].sum()
    )
    .reset_index(drop=True)
)

print(f"Overall length of q5_sentences_table: {len(q5_sentences_table)}")

Overall length of q5_sentences_table: 6248


In [26]:
q5_sentences_table.head(15).style.format(
    {"total_sentence_count": "{:,.0f}", "percentage": "{:.2%}"}
)

,character,total_sentence_count,percentage
0,Homer Simpson,"49,697",22.17%
1,Marge Simpson,"20,737",9.25%
2,Bart Simpson,"20,581",9.18%
3,Lisa Simpson,"16,998",7.58%
4,C. Montgomery Burns,"5,954",2.66%
5,Moe Szyslak,"5,521",2.46%
6,Seymour Skinner,"4,365",1.95%
7,Ned Flanders,"3,560",1.59%
8,Chief Wiggum,"3,409",1.52%
9,Krusty the Clown,"3,397",1.52%


In [27]:
q5_sentences_table.tail().style.format(
    {"total_sentence_count": "{:,.0f}", "percentage": "{:.2%}"}
)

,character,total_sentence_count,percentage
6243,Lancelot Link,1,0.00%
6244,Russian,1,0.00%
6245,Landscaper,1,0.00%
6246,Baby Bart,1,0.00%
6247,Nurse Dora,1,0.00%


In [28]:
# Q5 - iteration 2.1: horizontal bar chart
def get_top_n_sentences(n=20):
    return sentences_per_char.nlargest(n, "sentence_count")


n = 10
top_n = get_top_n_sentences(n)

bars = (
    alt.Chart(top_n)
    .mark_bar()
    .encode(
        x=alt.X("sentence_count:Q", title="Total Sentence Count"),
        y=alt.Y("character:N", sort="-x", title="Character"),
        tooltip=["character", "sentence_count"],
    )
)

labels = (
    alt.Chart(top_n)
    .mark_text(align="left", baseline="middle", dx=4)
    .encode(
        x="sentence_count:Q",
        y=alt.Y("character:N", sort="-x"),
        text=alt.Text("sentence_count:Q", format=","),
    )
)

(bars + labels).properties(
    title=f"Top {n} Characters by Total Sentence Count", width=500, height=400
)

alt.LayerChart(...)

In [29]:
# Q5 - iteration 2.2: vertical bar chart
n = 10
top_n = get_top_n_sentences(n)

bars = (
    alt.Chart(top_n)
    .mark_bar()
    .encode(
        x=alt.X(
            "character:N",
            sort="-y",
            title="Character",
            axis=alt.Axis(labelAngle=-45),
        ),
        y=alt.Y("sentence_count:Q", title="Total Sentence Count"),
        tooltip=["character", alt.Tooltip("sentence_count:Q", format=",")],
    )
)

bars.properties(
    title=f"Top {n} Characters by Total Sentence Count", width=600, height=400
)

alt.Chart(...)

In [30]:
# Q5 - iteration 3: lollipop chart
n = 20
base = alt.Chart(get_top_n_sentences(n)).encode(
    x=alt.X("sentence_count:Q", title="Total Sentence Count"),
    y=alt.Y("character:N", sort="-x", title="Character"),
    tooltip=["character", "sentence_count"],
)

points = base.mark_circle(size=80, color="#e45756")
lines = base.mark_rule(color="#e45756")

(lines + points).properties(
    title=f"Top {n} Characters by Total Sentence Count", width=500, height=400
)

alt.LayerChart(...)

### Final Visualization


In [31]:
# Q5 - final

### Design Discussion (max 200 words)

_TODO: Explain chart type choice, design process steps, changes for legibility, how interaction answers the question, and alternatives considered._


---

## Final Multi-View Visualization

This section combines all questions into a single cohesive dashboard with cross-interactions between views.


In [32]:
# Final multi-view visualization

### Design Discussion (max 200 words)

_TODO: Explain what changes were made to make charts consistent, aesthetically pleasant, color-consistent, and how cross-interactions work across views._
